# GenAI-Kickstart
## Lesson: Tools

Function or tool calling in Large Language Models (LLMs) is a process where the model interacts with predefined or external functions to extend its capabilities. Instead of just generating plain text, the LLM can call a specific function with structured data, perform computations, fetch real-time information, or interact with APIs. This mechanism bridges the gap between conversational AI and actionable tasks, making LLMs more interactive and useful in real-world applications.

During this lesson, you will learn:
* how to integrate tool calling

**Context**: In this lesson we continue with the digital bookstore theme from the prompt engineering lesson. You will build tools that search and order books from the bookstore catalog.

In [1]:
import sys
sys.path.append('../../../')

import json
import time
from pydantic import BaseModel
from litellm import completion
from dotenv import load_dotenv, find_dotenv
from lib.completion import generate_completion_with_tools
from lib.validation import validate_environment_for_model
from lib.constants import PROCESSED_DATA_PATH
import os

load_dotenv(find_dotenv())

True

In [2]:
# Models
GENERATION_PROVIDER = os.getenv("DEFAULT_GENERATION_PROVIDER")
GENERATION_MODEL = os.getenv("DEFAULT_GENERATION_MODEL")

# Generation parameters
DEFAULT_CONFIG_TEMPERATURE = 0.9
DEFAULT_CONFIG_MAX_OUTPUT_TOKENS = 500
DEFAULT_SYSTEM_PROMPT = "You are a helpful assistant. You can call functions to get information. If you need to call a function, please do so. If not, just answer the question. If you are not sure, ask for clarification."

# Set the full model name
generation_model = f"{GENERATION_PROVIDER}/{GENERATION_MODEL}"

# Validate environment
validate_environment_for_model(model=generation_model)

In [3]:
# Load the book catalog from the bookstore database
BOOK_DB_PATH = PROCESSED_DATA_PATH / "books.db"
with open(BOOK_DB_PATH) as f:
    book_catalog = json.load(f)

print(f"Loaded {len(book_catalog)} books from the catalog.")

Loaded 16 books from the catalog.


In [4]:
# Bookstore domain models and functions used in the exercises

class BookInfo(BaseModel):
    """Information about a book from the catalog."""
    title: str
    isbn: str
    pub_year: int
    summary: str
    url: str


def search_book_by_author(author: str) -> list[BookInfo]:
    """Search the bookstore catalog for books by a given author.

    Args:
        author: The name (or partial name) of the author to search for.

    Returns:
        A list of matching BookInfo objects.
    """
    print(f"Searching catalog for books by: '{author}'...")
    results = []
    for book in book_catalog:
        if author.lower() in book["summary"].lower():
            results.append(BookInfo(
                title=book["metadata"]["title"],
                isbn=book["metadata"]["isbn"],
                pub_year=book["metadata"]["pub_year"],
                summary=book["summary"],
                url=book["metadata"]["url"],
            ))
    print(f"Found {len(results)} result(s).")
    return results


def order_book(isbn: str) -> str:
    """Order a book by its ISBN number.

    Args:
        isbn: The ISBN of the book to order.

    Returns:
        A status message about the order.
    """
    print(f"Ordering book with ISBN: {isbn}")
    for _ in range(5):
        print(".", end="", flush=True)
        time.sleep(0.3)
    print()

    # Look up the book in the catalog
    for book in book_catalog:
        if book["metadata"]["isbn"] == isbn:
            status = f"Success: You ordered '{book['metadata']['title']}'!"
            print(status)
            return status

    status = f"Error: No book found with ISBN {isbn}."
    print(status)
    return status

### Exercise 01: Function calling
We can provide the model with tools in the form of functions. These functions are not executed directly by the model; instead, the model returns the function to be called along with the corresponding arguments, if it deems it necessary.  
In this exercise, you should define two prompts:
1. The first prompt should lead to the invocation of the `search_book_by_author` function shown below.
2. The second prompt should not be responded with a function call.

**Note**: LiteLLM uses the OpenAI tool format. Each tool is wrapped in `{"type": "function", "function": {...}}`.

In [5]:
# The declaration of a function in OpenAI tool format provided for the model
search_book_by_author_tool = {
    "type": "function",
    "function": {
        "name": "search_book_by_author",
        "description": "Search the bookstore catalog for all books by a given author. Returns matching books with title, ISBN, year and summary.",
        "parameters": {
            "type": "object",
            "properties": {
                "author": {
                    "type": "string",
                    "description": "The name (or partial name) of the author to search for (e.g., 'H. G. Wells', 'Stevenson', 'Conan Doyle').",
                },
            },
            "required": ["author"],
        },
    },
}

In [6]:
# TODO: Define a prompt which should be answered by the model with a function call
user_prompt = "What books by H. G. Wells do you have in your catalog?"

# Call the model with the tool declaration
response = generate_completion_with_tools(
    user_prompt=user_prompt,
    tools=[search_book_by_author_tool],
    model_name=generation_model,
    system_prompt=DEFAULT_SYSTEM_PROMPT,
    verbose=False,
)
response

{'text': None,
 'tool_calls': [ChatCompletionMessageToolCall(index=0, provider_specific_fields={'thought_signature': 'CscCAQw51seC5wiP1cf8t2OpWnZT0/b5M4WTX4Aq1GItFUhZdTZ+UY9RbNxcVAIvzYo2iWPs03+bKmWEaFsSrworh0QQSn4n3hSZrbAKEY9E6ALdJvUsmOyprWcl9I07Wqo8wZcr3AmQDj4cIgSxFAHiRR+06c3qefzwnIERMqtlZgwoavlVb+PkBqGBNKkBx7gL9JJnpVZhAfq355H5hchJH2yrhPtL6DkFqi7O5EhAvc02QYrLzpzP2jEai/mmvv8hfwwuKJR9stQGR/3qptY1neREHQ73Z7qxIflmg/TJiHwHIADDIBpaywFE2phiGR7gOvGAlMWqfsD0CIjb1CX+dT2GAB2yEoMyDkplahg3Ckm6Rz0s3YRDKB7eZAyqysYKbQ5Dd/Wiimh2VXSlizVCgZ7hzM8suX4kEkBTlJYb2NiP/BbXECAS'}, function=Function(arguments='{"author": "H. G. Wells"}', name='search_book_by_author'), id='call_f7c4852393904d7aa27aeeb70823__thought__CscCAQw51seC5wiP1cf8t2OpWnZT0/b5M4WTX4Aq1GItFUhZdTZ+UY9RbNxcVAIvzYo2iWPs03+bKmWEaFsSrworh0QQSn4n3hSZrbAKEY9E6ALdJvUsmOyprWcl9I07Wqo8wZcr3AmQDj4cIgSxFAHiRR+06c3qefzwnIERMqtlZgwoavlVb+PkBqGBNKkBx7gL9JJnpVZhAfq355H5hchJH2yrhPtL6DkFqi7O5EhAvc02QYrLzpzP2jEai/mmvv8hfwwuKJR9stQGR/3qptY1neREHQ73Z7qxIflmg/TJiH

In [7]:
# TODO: Define a prompt which should be answered by the model without a function call
user_prompt = "What is the meaning of life?"

response = generate_completion_with_tools(
    user_prompt=user_prompt,
    tools=[search_book_by_author_tool],
    model_name=generation_model,
    system_prompt=DEFAULT_SYSTEM_PROMPT,
    verbose=False,
)
response

{'text': 'That is a profound philosophical question that is beyond my current capabilities to answer. I can help you with tasks like searching for books by a particular author.',
 'tool_calls': None}

### Exercise 02: Execute function call
Below you see the `order_book` function. Your task is to...
1. Provide a JSON description for this function to the model (in OpenAI tool format). You do not need to provide the return type.
2. Create a prompt which should trigger this function
3. Automatically trigger this function depending on the response of the model

In [8]:
# The order_book function is already defined above. Here is a reminder of its signature:
# def order_book(isbn: str) -> str:
#     """Order a book by its ISBN number."""
#
# It looks up the ISBN in the catalog and returns a status message.
# Try it out:
order_book("978-3-51593-12345-6")

Ordering book with ISBN: 978-3-51593-12345-6
.....
Success: You ordered 'A Study in Scarlet'!


"Success: You ordered 'A Study in Scarlet'!"

In [9]:
# TODO: Define the tool declaration for the model (OpenAI format)
order_book_tool = {
    "type": "function",
    "function": {
        "name": "order_book",
        "description": "Order a book from the bookstore by its ISBN number.",
        "parameters": {
            "type": "object",
            "properties": {
                "isbn": {
                    "type": "string",
                    "description": "The ISBN of the book to order (e.g., '978-3-51593-12345-6').",
                }
            },
            "required": ["isbn"],
        },
    },
}

In [10]:
# TODO: Define a prompt which should trigger the order_book function
user_prompt = "Please order the book with ISBN 978-3-51593-12345-6."

# Call the model with the tool declaration
response = generate_completion_with_tools(
    user_prompt=user_prompt,
    tools=[order_book_tool],
    model_name=generation_model,
    system_prompt=DEFAULT_SYSTEM_PROMPT,
    verbose=False,
)

# Check if the model provided a tool call
if response["tool_calls"]:
    tool_call = response["tool_calls"][0]
    function_name = tool_call.function.name
    function_args = json.loads(tool_call.function.arguments)

    # Call the function dynamically
    if function_name == "order_book":
        status = order_book(**function_args)
    else:
        print(f"Function {function_name} not recognized.")
else:
    print(f"Response: {response['text']}")

Ordering book with ISBN: 978-3-51593-12345-6
.....
Success: You ordered 'A Study in Scarlet'!


### (Optional) Exercise 03: Full circle - Interactive bookstore assistant
In exercises 01 and 02 we called a tool but never fed the result back to the model. In a real assistant the flow is a **loop**:

1. The user sends a message
2. The model responds — either with **text** or with a **tool call**
3. If tool call → execute the function locally → append the result as a `tool` message → call the model again (go to 2)
4. If text → show it to the user → wait for next input (go to 1)

Your task is to implement this loop using both `search_book_by_author_tool` and `order_book_tool`.  
The model decides *on its own* which tool to call (or whether to just answer with text).

**Hints**
* After executing a tool call, feed the result back by appending a message with `role: "tool"`, `tool_call_id` (from the tool call object) and the result as `content` (string).
* The model's response message (including tool calls) must also be appended to the message history — use `message.model_dump()` to serialize it.
* Use `json.dumps()` to convert complex return values (like lists of BookInfo) to a string before feeding them back.

In [11]:
# Map of available functions for dynamic dispatch
available_functions = {
    "search_book_by_author": search_book_by_author,
    "order_book": order_book,
}

# All tools the assistant can use
tools = [search_book_by_author_tool, order_book_tool]


def execute_tool_call(tool_call) -> str:
    """Execute a tool call and return the result as a string."""
    function_name = tool_call.function.name
    function_args = json.loads(tool_call.function.arguments)

    if function_name not in available_functions:
        return f"Error: Function {function_name} not recognized."

    result = available_functions[function_name](**function_args)

    # Convert result to a JSON string so the model can read it
    if isinstance(result, list):
        return json.dumps([r.model_dump() for r in result])
    return str(result)


# TODO: Implement the interactive bookstore assistant loop
def bookstore_assistant():
    """Interactive bookstore assistant with a tool-calling loop."""
    messages = [{"role": "system", "content": DEFAULT_SYSTEM_PROMPT}]

    print("Welcome to the bookstore! Type 'exit' to quit.\n")

    while True:
        user_input = input("You:\n")
        if user_input.lower() == "exit":
            print("Goodbye!")
            break

        # Append the user message to the conversation history
        messages.append({"role": "user", "content": user_input})

        # TODO: Inner loop — keep calling the model until it responds with text (not a tool call)
        while True:
            response = completion(
                model=generation_model,
                messages=messages,
                tools=tools,
            )
            message = response.choices[0].message

            # TODO: Append the assistant's response to the history
            messages.append(message.model_dump())

            # TODO: Check if the model wants to call a tool
            if message.tool_calls:
                # TODO: Execute each tool call and feed the result back
                for tool_call in message.tool_calls:
                    result_str = execute_tool_call(tool_call)
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": result_str,
                    })
                # Continue the inner loop — the model will now respond based on the tool results
            else:
                # The model responded with text — show it and break back to user input
                print(f"\nAssistant:\n{message.content}\n")
                break

In [13]:
# Start the interactive bookstore assistant
# Try a conversation like:
#   "What books by H. G. Wells do you have?"
#   "Please order The Time Machine"
#   "exit"
bookstore_assistant()

Welcome to the bookstore! Type 'exit' to quit.

Searching catalog for books by: 'H. G. Wells'...
Found 2 result(s).

Assistant:
We have the following books by H. G. Wells:

*   **The Time Machine** (ISBN: 978-3-46816-12345-6, Published: 2004)
*   **The War of the Worlds** (ISBN: 978-3-42306-12345-6, Published: 2004)

Goodbye!
